# RAG Step 3 — Retrieving Relevant Context

## 1. Install and import libraries

In [1]:
!pip -q install chromadb

import chromadb
print("Libraries ready.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 778.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/1

## 2. Recreate the vector collection

In [2]:
client = chromadb.Client()
collection = client.get_or_create_collection(name="rag_retrieval_demo")
documents = [
    "RAG combines information retrieval with language generation.",
    "A RAG system stores documents in a searchable form.",
    "Relevant pieces of documents are retrieved when a user asks a question.",
    "Retrieved context is provided to a language model to generate a grounded answer.",
    "Chunking helps large documents become easier to retrieve efficiently."
]
collection.upsert(documents=documents, ids=[f"doc_{i}" for i in range(len(documents))])
print("Documents stored:", collection.count())

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 11.1MiB/s]


Documents stored: 5


## 3. Retrieve relevant documents

In [3]:
query = "How does a RAG system find relevant information?"
results = collection.query(query_texts=[query], n_results=3)
print("Query:", query)
print("\nRetrieved documents:")
for i, doc in enumerate(results["documents"][0], 1):
    print(f"{i}. {doc}")

Query: How does a RAG system find relevant information?

Retrieved documents:
1. A RAG system stores documents in a searchable form.
2. RAG combines information retrieval with language generation.
3. Retrieved context is provided to a language model to generate a grounded answer.


## 4. View distances

In [4]:
print("Retrieved distances (smaller is closer):")
for i, distance in enumerate(results["distances"][0], 1):
    print(f"Result {i}: {distance:.4f}")

Retrieved distances (smaller is closer):
Result 1: 0.6014
Result 2: 0.9863
Result 3: 1.5312


## 5. Build context for the language model

In [5]:
context = "\n\n".join(results["documents"][0])
print("Context prepared for the LLM:\n")
print(context)

Context prepared for the LLM:

A RAG system stores documents in a searchable form.

RAG combines information retrieval with language generation.

Retrieved context is provided to a language model to generate a grounded answer.


## 6. Example grounded response

In [6]:
answer = (
    "A RAG system finds relevant information by storing documents in a searchable "
    "form and retrieving the most relevant pieces when a user asks a question. "
    "That retrieved context can then be supplied to a language model."
)
print("Question:", query)
print("\nExample answer:", answer)
print("\nNote: This answer is hard-coded; it is not generated by an LLM API call.")

Question: How does a RAG system find relevant information?

Example answer: A RAG system finds relevant information by storing documents in a searchable form and retrieving the most relevant pieces when a user asks a question. That retrieved context can then be supplied to a language model.

Note: This answer is hard-coded; it is not generated by an LLM API call.


## 7. Final status

In [7]:
print("Retrieval: completed")
print("Retrieved chunks:", len(results["documents"][0]))
print("Context preparation: completed")
print("RAG Step 3 completed successfully.")

Retrieval: completed
Retrieved chunks: 3
Context preparation: completed
RAG Step 3 completed successfully.
